In [1]:
import numpy as np
from numba import jit, njit

## Part I

Дан массив A размера (X, Y, Z) -- к примеру, (1000, 1000, 30). Отдельный элемент A[i, j, :] назовем трассой из A.
дан второй массив B размера (N, Z) -- несколько трасс. Необходимо для каждой трассы из A посчитать среднее значение корреляции между A[i, j, :] и всеми трассами из B. Примерное значение N -- от 20 до 100


Some notes

In [2]:
A = np.random.randn(1000, 1000, 30)

In [3]:
%%timeit
u = np.zeros_like(A)

51.5 ms ± 185 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [4]:
%%timeit
u = np.zeros((A.shape[:2]))

294 µs ± 1.38 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [5]:
%%timeit
u = np.empty_like(A)

6.9 µs ± 321 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [6]:
%%timeit
u = np.empty((A.shape[:2]))

584 ns ± 2.53 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [7]:
x = np.random.randn(1000000)
y = np.random.randn(1000000) 

In [8]:
%%time
x @ y 

CPU times: user 9.39 ms, sys: 12.4 ms, total: 21.8 ms
Wall time: 635 µs


-466.1678814645154

In [9]:
%%time
(x * y).sum()

CPU times: user 27.7 ms, sys: 255 ms, total: 282 ms
Wall time: 4.42 ms


-466.16788146451637

### Solution via for loop (slow)

In [10]:
def exchange_correlation_slow(A, B):
    """ Origin function """
    X, Y, Z = A.shape # 1000 x 1000 x 30
    mean_corr_for_traces = np.empty((X, Y))
    B_normalized = (B - B.mean(axis=-1, keepdims=True)) / B.std(axis=-1, keepdims=True)
    
    for i in range(X):
        for j in range(Y):
            res = np.mean((A[i, j] - A[i, j].mean()) * B_normalized, axis=-1) / A[i, j].std()
            mean_corr_for_traces[i, j] = np.mean(res)
    
    return mean_corr_for_traces

In [11]:
def exchange_correlation_slow_trace(A, B):
    """ Call __getitem__() only once for each trace in for loop """
    X, Y, Z = A.shape # 1000 x 1000 x 30
    mean_corr_for_traces = np.empty((X, Y))
    B_normalized = (B - B.mean(axis=-1, keepdims=True)) / B.std(axis=-1, keepdims=True)
    
    for i in range(X):
        for j in range(Y):
            trace = A[i, j]
            res = np.mean((trace - trace.mean()) * B_normalized, axis=-1) / trace.std()
            mean_corr_for_traces[i, j] = np.mean(res)
    
    return mean_corr_for_traces

In [12]:
def exchange_correlation_slow_prenormalized(A, B):
    """ Pre-normalize matrix A """
    X, Y, Z = A.shape # 1000 x 1000 x 30
    mean_corr_for_traces = np.empty((X, Y))
    B_normalized = (B - B.mean(axis=-1, keepdims=True)) / B.std(axis=-1, keepdims=True)
    A_normalized = (A - A.mean(axis=-1, keepdims=True)) / A.std(axis=-1, keepdims=True)
    
    for i in range(X):
        for j in range(Y):
            trace = A_normalized[i, j]
            res = np.mean(trace * B_normalized, axis=-1)
            mean_corr_for_traces[i, j] = np.mean(res)
    
    return mean_corr_for_traces

In [13]:
def exchange_correlation_slow_trace_matmul(A, B):
    """ Matrix mul in for loop """
    X, Y, Z = A.shape # 1000 x 1000 x 30
    mean_corr_for_traces = np.empty((X, Y))
    B_normalized = (B - B.mean(axis=-1, keepdims=True)) / B.std(axis=-1, keepdims=True)
    A_normalized = (A - A.mean(axis=-1, keepdims=True)) / A.std(axis=-1, keepdims=True)
    
    for i in range(X):
        for j in range(Y):
            trace = A_normalized[i, j]
            res = 1 / Z * trace @ B_normalized.T
            mean_corr_for_traces[i, j] = np.mean(res)
    
    return mean_corr_for_traces

In [14]:
@njit
def exchange_correlation_numba(A, B):
    """ Use numba. Since numba doesn't support np.mean(axis=-1), mean was realized via for loop """
    X, Y, Z = A.shape # 1000 x 1000 x 30
    N = B.shape[0]
    
    B_mean = np.empty(N)
    B_std = np.empty(N)
    for i in range(N):
        B_mean[i] = np.mean(B[i, :])
        B_std[i] = np.std(B[i, :])
    
    B_mean = B_mean.reshape(-1, 1)
    B_std = B_std.reshape(-1, 1)
    B_normalized = (B - B_mean) / B_std
    
    mean_corr_for_traces = np.empty((X, Y))
    for i in range(X):
        for j in range(Y):
            trace = A[i, j]
            trace_normalized = (trace - trace.mean()) / trace.std()
            mul = trace_normalized * B_normalized
            
            running_mean = np.empty(N)
            for k in range(N):
                running_mean[k] = np.mean(mul[k, :])
    
            mean_corr_for_traces[i, j] = np.mean(running_mean)
    
    return mean_corr_for_traces

In [15]:
def exchange_correlation_without_numba(A, B):
    """ The same function as in the numba case but without numba njit decorator """
    X, Y, Z = A.shape # 1000 x 1000 x 30
    N = B.shape[0]
    
    B_mean = np.empty(B.shape[0])
    B_std = np.empty(B.shape[0])
    for i in range(B.shape[0]):
        B_mean[i] = np.mean(B[i, :])
        B_std[i] = np.std(B[i, :])
    
    B_mean = B_mean.reshape(-1, 1)
    B_std = B_std.reshape(-1, 1)
    B_normalized = (B - B_mean) / B_std
    
    mean_corr_for_traces = np.empty((X, Y))
    for i in range(X):
        for j in range(Y):
            trace = A[i, j]
            trace_normalized = (trace - trace.mean()) / trace.std()
            mul = trace_normalized * B_normalized
            
            running_mean = np.empty(N)
            for k in range(N):
                running_mean[k] = np.mean(mul[k, :])
    
            mean_corr_for_traces[i, j] = np.mean(running_mean)
    
    return mean_corr_for_traces

### Solution via matrix mul (fast)

In [16]:
def exchange_correlation(A, B):
    """ For each trace in the trace matrix A computes mean correlation with each trace in trace matrix B
    
    Parameters 
    ----------
    A: np.ndarray, shape = (X, Y, Z)
        First trace matrix. Matrix A has X * Y traces.
    B: np.ndarray, shape = (N, Z)
        Second trace matrix.
        
    Returns
    -------
    correlation_matrix: np.ndarray, shape = (X, Y)
        Matrix with mean values of correlation for each trace in A. 
    """
    X, Y, Z = A.shape # 1000 x 1000 x 30
    A_reshaped = A.reshape(X * Y, -1)
    
    # normalize matrices to compute correaltions
    A_normalized = (A_reshaped - A_reshaped.mean(axis=-1, keepdims=True)) / A_reshaped.std(axis=-1, keepdims=True)
    B_normalized = (B - B.mean(axis=-1, keepdims=True)) / B.std(axis=-1, keepdims=True)
    
    # compute correlation for each trace in A matrix (we have X * Y traces)
    correlation_matrix = np.mean(1 / Z * A_normalized @ B_normalized.T, axis=-1).reshape(X, -1) # shape = (X, Y)
    
    return correlation_matrix

## Trials

In [17]:
A = np.random.randn(1000, 1000, 30)
#A = np.random.randn(100, 100, 30)

### Let's compare slow and fast solutions with N = 20

In [18]:
N = 20
B = np.random.randn(N, 30)

In [19]:
%%time 
output_slow = exchange_correlation_slow(A, B)
print(output_slow.shape)
output_slow

(1000, 1000)
CPU times: user 39.1 s, sys: 99.9 ms, total: 39.2 s
Wall time: 39.2 s


array([[ 0.04205609, -0.02024399,  0.08831388, ...,  0.0107079 ,
        -0.01361813, -0.06158321],
       [-0.02178272, -0.0242575 ,  0.01412216, ...,  0.00891786,
        -0.04940107, -0.05034677],
       [-0.05623285,  0.13017262,  0.02369724, ..., -0.07166375,
        -0.03660323,  0.0084345 ],
       ...,
       [-0.08121465,  0.07297323,  0.01439966, ...,  0.04858886,
        -0.03522883,  0.01399171],
       [-0.06491935, -0.08398215, -0.0303899 , ...,  0.04882211,
         0.00735098,  0.06450512],
       [ 0.01439838, -0.03427303,  0.01149061, ..., -0.0249336 ,
        -0.00248467, -0.02201403]])

In [20]:
%%time
output_slow_trace = exchange_correlation_slow_trace(A, B)
print(output_slow_trace.shape)
output_slow_trace

(1000, 1000)
CPU times: user 38.9 s, sys: 30.2 ms, total: 38.9 s
Wall time: 38.9 s


array([[ 0.04205609, -0.02024399,  0.08831388, ...,  0.0107079 ,
        -0.01361813, -0.06158321],
       [-0.02178272, -0.0242575 ,  0.01412216, ...,  0.00891786,
        -0.04940107, -0.05034677],
       [-0.05623285,  0.13017262,  0.02369724, ..., -0.07166375,
        -0.03660323,  0.0084345 ],
       ...,
       [-0.08121465,  0.07297323,  0.01439966, ...,  0.04858886,
        -0.03522883,  0.01399171],
       [-0.06491935, -0.08398215, -0.0303899 , ...,  0.04882211,
         0.00735098,  0.06450512],
       [ 0.01439838, -0.03427303,  0.01149061, ..., -0.0249336 ,
        -0.00248467, -0.02201403]])

In [21]:
%%time
output_slow_trace_matmul = exchange_correlation_slow_trace_matmul(A, B)
print(output_slow_trace_matmul.shape)
output_slow_trace_matmul

(1000, 1000)
CPU times: user 9.41 s, sys: 116 ms, total: 9.53 s
Wall time: 9.53 s


array([[ 0.04205609, -0.02024399,  0.08831388, ...,  0.0107079 ,
        -0.01361813, -0.06158321],
       [-0.02178272, -0.0242575 ,  0.01412216, ...,  0.00891786,
        -0.04940107, -0.05034677],
       [-0.05623285,  0.13017262,  0.02369724, ..., -0.07166375,
        -0.03660323,  0.0084345 ],
       ...,
       [-0.08121465,  0.07297323,  0.01439966, ...,  0.04858886,
        -0.03522883,  0.01399171],
       [-0.06491935, -0.08398215, -0.0303899 , ...,  0.04882211,
         0.00735098,  0.06450512],
       [ 0.01439838, -0.03427303,  0.01149061, ..., -0.0249336 ,
        -0.00248467, -0.02201403]])

In [22]:
%%time
output_slow_prenormalized = exchange_correlation_slow_prenormalized(A, B)
print(output_slow_prenormalized.shape)
output_slow_prenormalized

(1000, 1000)
CPU times: user 16.5 s, sys: 140 ms, total: 16.6 s
Wall time: 16.6 s


array([[ 0.04205609, -0.02024399,  0.08831388, ...,  0.0107079 ,
        -0.01361813, -0.06158321],
       [-0.02178272, -0.0242575 ,  0.01412216, ...,  0.00891786,
        -0.04940107, -0.05034677],
       [-0.05623285,  0.13017262,  0.02369724, ..., -0.07166375,
        -0.03660323,  0.0084345 ],
       ...,
       [-0.08121465,  0.07297323,  0.01439966, ...,  0.04858886,
        -0.03522883,  0.01399171],
       [-0.06491935, -0.08398215, -0.0303899 , ...,  0.04882211,
         0.00735098,  0.06450512],
       [ 0.01439838, -0.03427303,  0.01149061, ..., -0.0249336 ,
        -0.00248467, -0.02201403]])

In [23]:
%%time
output_fast = exchange_correlation(A, B)
print(output_fast.shape)
output_fast

(1000, 1000)
CPU times: user 2.61 s, sys: 3.23 s, total: 5.84 s
Wall time: 489 ms


array([[ 0.04205609, -0.02024399,  0.08831388, ...,  0.0107079 ,
        -0.01361813, -0.06158321],
       [-0.02178272, -0.0242575 ,  0.01412216, ...,  0.00891786,
        -0.04940107, -0.05034677],
       [-0.05623285,  0.13017262,  0.02369724, ..., -0.07166375,
        -0.03660323,  0.0084345 ],
       ...,
       [-0.08121465,  0.07297323,  0.01439966, ...,  0.04858886,
        -0.03522883,  0.01399171],
       [-0.06491935, -0.08398215, -0.0303899 , ...,  0.04882211,
         0.00735098,  0.06450512],
       [ 0.01439838, -0.03427303,  0.01149061, ..., -0.0249336 ,
        -0.00248467, -0.02201403]])

In [24]:
%%timeit
output_numba = exchange_correlation_numba(A, B) 

1.05 s ± 4.67 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [25]:
output_numba = exchange_correlation_numba(A, B)
print(output_numba.shape)
output_numba 

(1000, 1000)


array([[ 0.04205609, -0.02024399,  0.08831388, ...,  0.0107079 ,
        -0.01361813, -0.06158321],
       [-0.02178272, -0.0242575 ,  0.01412216, ...,  0.00891786,
        -0.04940107, -0.05034677],
       [-0.05623285,  0.13017262,  0.02369724, ..., -0.07166375,
        -0.03660323,  0.0084345 ],
       ...,
       [-0.08121465,  0.07297323,  0.01439966, ...,  0.04858886,
        -0.03522883,  0.01399171],
       [-0.06491935, -0.08398215, -0.0303899 , ...,  0.04882211,
         0.00735098,  0.06450512],
       [ 0.01439838, -0.03427303,  0.01149061, ..., -0.0249336 ,
        -0.00248467, -0.02201403]])

In [26]:
%%time
output_without_numba = exchange_correlation_without_numba(A, B)

CPU times: user 2min 27s, sys: 1.35 s, total: 2min 29s
Wall time: 2min 27s


In [27]:
print(output_without_numba.shape)
output_without_numba

(1000, 1000)


array([[ 0.04205609, -0.02024399,  0.08831388, ...,  0.0107079 ,
        -0.01361813, -0.06158321],
       [-0.02178272, -0.0242575 ,  0.01412216, ...,  0.00891786,
        -0.04940107, -0.05034677],
       [-0.05623285,  0.13017262,  0.02369724, ..., -0.07166375,
        -0.03660323,  0.0084345 ],
       ...,
       [-0.08121465,  0.07297323,  0.01439966, ...,  0.04858886,
        -0.03522883,  0.01399171],
       [-0.06491935, -0.08398215, -0.0303899 , ...,  0.04882211,
         0.00735098,  0.06450512],
       [ 0.01439838, -0.03427303,  0.01149061, ..., -0.0249336 ,
        -0.00248467, -0.02201403]])

In [28]:
np.allclose(output_slow, output_fast), \
np.allclose(output_slow_trace_matmul, output_fast), \
np.allclose(output_slow_prenormalized, output_fast), \
np.allclose(output_numba, output_fast), \
np.allclose(output_without_numba, output_fast)

(True, True, True, True, True)

### Выводы
Если в цикле for обращаться к каждый трассе один раз, т.е. trace = A[i, j] --> работает быстрее

Если сделать нормировку матрицы A перед циклом, а не для каждой трассы на каждой итерации --> работает быстрее

Если сделать расчет корреляции через матричное умножение, а не через умножение матриц с бродкастом --> работает быстрее

Numba значительно выйгрывает в скорости в решении через циклы

### Основной вывод
Значительное замедление по сравнению с полностью векторизованным решением дает итерирование с помощью цикла for

Numba дает хороший буст в скорости

### Fast solution with different N

In [29]:
%%time
N = 50 
B = np.random.randn(N, 30)
output = exchange_correlation(A, B)
print(output.shape)
output

(1000, 1000)
CPU times: user 5.88 s, sys: 4.96 s, total: 10.8 s
Wall time: 571 ms


array([[ 0.03060016, -0.00328573, -0.04298524, ..., -0.00137033,
         0.00404348, -0.00460347],
       [-0.01120051, -0.0367292 , -0.00953309, ...,  0.00724785,
         0.01562242, -0.04203089],
       [ 0.05972498,  0.04317005, -0.00390266, ...,  0.00884383,
         0.03067482,  0.00233498],
       ...,
       [ 0.01422341, -0.00620201, -0.04245995, ..., -0.0382429 ,
         0.01162831, -0.01863173],
       [ 0.00823209,  0.00190509, -0.01642379, ..., -0.02416892,
         0.02564531,  0.04568355],
       [-0.04141315, -0.03605563, -0.010212  , ..., -0.00715431,
        -0.00154342, -0.00718346]])

In [30]:
%%time
N = 70 
B = np.random.randn(N, 30)
output = exchange_correlation(A, B)
print(output.shape)
output

(1000, 1000)
CPU times: user 9.06 s, sys: 8.86 s, total: 17.9 s
Wall time: 661 ms


array([[ 0.01314641, -0.05085116,  0.01578392, ...,  0.0085585 ,
        -0.02885551, -0.00556651],
       [ 0.02805694,  0.00499917, -0.0136196 , ...,  0.00058   ,
        -0.00147742, -0.01197484],
       [ 0.00652743,  0.03678747,  0.01233875, ...,  0.01545896,
        -0.02224355,  0.03269147],
       ...,
       [ 0.00670641, -0.0085959 ,  0.01884643, ..., -0.00793879,
        -0.05312186,  0.01210008],
       [-0.0094139 ,  0.00794121, -0.00103116, ...,  0.04519633,
        -0.01307197,  0.01307496],
       [ 0.00344382,  0.04011289,  0.00072269, ..., -0.01298507,
        -0.00583475, -0.03016353]])

In [31]:
%%time
N = 100 
B = np.random.randn(N, 30)
output = exchange_correlation(A, B)
print(output.shape)
output

(1000, 1000)
CPU times: user 9.51 s, sys: 10.5 s, total: 20 s
Wall time: 723 ms


array([[ 0.03431709, -0.02376573,  0.01789326, ...,  0.00631644,
        -0.01844379, -0.00645047],
       [ 0.00422277, -0.01614476,  0.02342456, ...,  0.01061467,
         0.01177171,  0.01957883],
       [-0.00291312,  0.01784197,  0.03319044, ...,  0.01493155,
        -0.00571111, -0.00648561],
       ...,
       [-0.03088646, -0.00471352, -0.04454988, ...,  0.00133384,
         0.0014582 , -0.00952039],
       [-0.00942508, -0.01016293, -0.0331646 , ...,  0.01298152,
        -0.00440125, -0.02395346],
       [ 0.00756977, -0.03476758, -0.00365766, ...,  0.00796025,
         0.03100252,  0.01225493]])

## Part II

Для квадратного окна заданного (нечетного) размера K посчитать массив output размера (X, Y), где на i,j месте находится среднее значение корреляции между трассой A[i, j, :] и всеми соседними трассами, лежащими внутри квадратного окна KxK с центром в i,j трассе. 
Примерное значение K -- от 3 до 13

### Slow solution 

In [32]:
def window_correlation(A, K):
    """ Applies sliding window to i, j trace in trace matrix A via np.corrcoef. """
    X, Y = A.shape[:2]
    output = np.empty((X, Y))
    
    for i in range(X):
        for j in range(Y):
            
            if i == 0 or i == X - 1:
                K_x = K - 1
            else: 
                K_x = K
                
            if j == 0 or j == Y - 1:
                K_y = K - 1
            else:
                K_y = K
                
            sum_corr = 0
            values_in_window = 0
            start_idx_j = max(0, j - K_y // 2)
            final_idx_j = min(Y, j + K_y // 2 + 1)
            start_idx_i = max(0, i - K_x // 2)
            final_idx_i = min(X, i + K_x // 2 + 1) 
            
            for window_i in range(start_idx_i, final_idx_i):
                for window_j in range(start_idx_j, final_idx_j):
                    
                    if i == window_i and j == window_j: # don't consider the central trace with itself 
                        continue

                    sum_corr += np.corrcoef(A[i, j], A[window_i, window_j])[0, 1]
                    values_in_window += 1
                    
            avg_corr = sum_corr / values_in_window
            
            output[i, j] = avg_corr
    
    return output 

In [33]:
def window_correlation_handy_corr(A, K):
    """ Applies sliding window to i, j trace in trace matrix A via handy correlations. """
    X, Y = A.shape[:2]
    output = np.empty((X, Y))
    
    for i in range(X):
        for j in range(Y):
            
            if i == 0 or i == X - 1:
                K_x = K - 1
            else: 
                K_x = K
                
            if j == 0 or j == Y - 1:
                K_y = K - 1
            else:
                K_y = K
                
            sum_corr = 0
            values_in_window = 0
            start_idx_j = max(0, j - K_y // 2)
            final_idx_j = min(Y, j + K_y // 2 + 1)
            start_idx_i = max(0, i - K_x // 2)
            final_idx_i = min(X, i + K_x // 2 + 1)
            curr_trace = A[i, j]
            
            for window_i in range(start_idx_i, final_idx_i):
                for window_j in range(start_idx_j, final_idx_j):
                    
                    if i == window_i and j == window_j: # don't consider the central trace with itself 
                        continue
                        
                    window_trace = A[window_i, window_j]
                    sum_corr += np.mean((curr_trace - curr_trace.mean()) * (window_trace - window_trace.mean())) \
                                / (curr_trace.std() * window_trace.std())
                    values_in_window += 1
                    
            avg_corr = sum_corr / values_in_window
            
            output[i, j] = avg_corr
    
    return output 

In [34]:
@njit
def window_correlation_handy_corr_numba(A, K):
    """ Applies sliding window to i, j trace in trace matrix A via handy correlations with numba. """
    X, Y = A.shape[:2]
    output = np.empty((X, Y))
    
    for i in range(X):
        for j in range(Y):
            
            if i == 0 or i == X - 1:
                K_x = K - 1
            else: 
                K_x = K
                
            if j == 0 or j == Y - 1:
                K_y = K - 1
            else:
                K_y = K
                
            sum_corr = 0
            values_in_window = 0
            start_idx_j = max(0, j - K_y // 2)
            final_idx_j = min(Y, j + K_y // 2 + 1)
            start_idx_i = max(0, i - K_x // 2)
            final_idx_i = min(X, i + K_x // 2 + 1) 
            curr_trace = A[i, j]
            
            for window_i in range(start_idx_i, final_idx_i):
                for window_j in range(start_idx_j, final_idx_j):
                    
                    if i == window_i and j == window_j: # don't consider the central trace with itself 
                        continue
                        
                    window_trace = A[window_i, window_j]    
                    sum_corr += np.mean((curr_trace - curr_trace.mean()) * (window_trace - window_trace.mean())) \
                                / (curr_trace.std() * window_trace.std())
                    values_in_window += 1
                    
            avg_corr = sum_corr / values_in_window
            
            output[i, j] = avg_corr
    
    return output 

### Fast solution

In [35]:
def trace_correlation(A, K):
    """ For window KxK computes mean correlation for trace in the middle of the window and its neighbours. 
    
    Parameters
    ----------
    A: np.ndarray, shape = (X, Y, Z)
        Trace matrix
    K: int, odd number
        Size of the kernel 
        
    Returns 
    -------
    mean_corrs: np.ndarray, shape = (X, Y)
       Matrix with mean correlation for each trace in the i, j position   
    """
    # other parameters
    padding = K // 2 
    mid = K ** 2 // 2 # idx of mid trace 
    X, Y, Z = A.shape # 1000 x 1000 x 30 
    
    # add padding to the trace matrix 
    A_pad = np.pad(A, pad_width=((padding, padding), (padding, padding), (0, 0)))
    
    # make K x K x Z patches
    patches = np.lib.stride_tricks.sliding_window_view(A_pad, window_shape=(K, K, Z)) # shape = (X, Y, 1, K, K, Z)
    patches_reshaped = patches.reshape(patches.shape[0] * patches.shape[1],
                                       patches.shape[2] * patches.shape[3] * patches.shape[4],
                                       patches.shape[5]) # shape = (X * Y, K * K, Z)
    
    # normalize and transpose patches to compute correlations
    patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \
                    / patches_reshaped.std(axis=-1, keepdims=True) 
    patches_norm_transposed = patches_norm.transpose((0, 2, 1))
    
    # find correlations for each trace in each patch and choose only mid trace
    cor_matrix = 1 / Z * (patches_norm @ patches_norm_transposed)[:, mid, :]
    cor_matrix[:, mid] = np.nan # replace ones to nans so that don't consider trace with itslef  
    mean_corrs = np.nanmean(cor_matrix, axis=1).reshape(X, -1) # compute average correlation for each trace
    
    return mean_corrs

## Trials

In [36]:
A = np.random.randn(1000, 1000, 30)

### Let's compare the functions on K = 3 and K = 5 

In [37]:
%%time
K = 3
output_slow_1 = window_correlation(A, K)
print(output_slow_1.shape)
output_slow_1

(1000, 1000)
CPU times: user 8min 24s, sys: 0 ns, total: 8min 24s
Wall time: 8min 24s


array([[-0.11346234, -0.00546136, -0.0775199 , ...,  0.0296181 ,
        -0.06386039, -0.04199768],
       [ 0.08667716, -0.08825905, -0.06856663, ...,  0.05598235,
        -0.06165052, -0.04068725],
       [ 0.10111272,  0.01626349, -0.05014453, ...,  0.09632763,
         0.04231648, -0.01214947],
       ...,
       [-0.07754684, -0.00684806,  0.14310739, ..., -0.13521372,
        -0.09376252, -0.026759  ],
       [ 0.03269487, -0.026429  , -0.05316353, ...,  0.06118422,
        -0.1858849 , -0.06747157],
       [-0.02768124, -0.03653215, -0.00473706, ..., -0.03963701,
        -0.0155793 , -0.14759911]])

In [38]:
%%time
K = 3
output_slow_handy_1 = window_correlation_handy_corr(A, K)
print(output_slow_handy_1.shape)
output_slow_handy_1

(1000, 1000)
CPU times: user 6min 27s, sys: 0 ns, total: 6min 27s
Wall time: 6min 27s


array([[-0.11346234, -0.00546136, -0.0775199 , ...,  0.0296181 ,
        -0.06386039, -0.04199768],
       [ 0.08667716, -0.08825905, -0.06856663, ...,  0.05598235,
        -0.06165052, -0.04068725],
       [ 0.10111272,  0.01626349, -0.05014453, ...,  0.09632763,
         0.04231648, -0.01214947],
       ...,
       [-0.07754684, -0.00684806,  0.14310739, ..., -0.13521372,
        -0.09376252, -0.026759  ],
       [ 0.03269487, -0.026429  , -0.05316353, ...,  0.06118422,
        -0.1858849 , -0.06747157],
       [-0.02768124, -0.03653215, -0.00473706, ..., -0.03963701,
        -0.0155793 , -0.14759911]])

In [39]:
%%timeit
K = 3
output_slow_handy_numba_1 = window_correlation_handy_corr_numba(A, K)

2.37 s ± 17.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [40]:
output_slow_handy_numba_1 = window_correlation_handy_corr_numba(A, K)
print(output_slow_handy_numba_1.shape)
output_slow_handy_numba_1

(1000, 1000)


array([[-0.11346234, -0.00546136, -0.0775199 , ...,  0.0296181 ,
        -0.06386039, -0.04199768],
       [ 0.08667716, -0.08825905, -0.06856663, ...,  0.05598235,
        -0.06165052, -0.04068725],
       [ 0.10111272,  0.01626349, -0.05014453, ...,  0.09632763,
         0.04231648, -0.01214947],
       ...,
       [-0.07754684, -0.00684806,  0.14310739, ..., -0.13521372,
        -0.09376252, -0.026759  ],
       [ 0.03269487, -0.026429  , -0.05316353, ...,  0.06118422,
        -0.1858849 , -0.06747157],
       [-0.02768124, -0.03653215, -0.00473706, ..., -0.03963701,
        -0.0155793 , -0.14759911]])

In [41]:
%%time
K = 3
output_fast_1 = trace_correlation(A=A, K=K)
print(output_fast_1.shape)
output_fast_1

/tmp/ipykernel_73086/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 2.91 s, sys: 1.64 s, total: 4.55 s
Wall time: 4.55 s


array([[-0.11346234, -0.00546136, -0.0775199 , ...,  0.0296181 ,
        -0.06386039, -0.04199768],
       [ 0.08667716, -0.08825905, -0.06856663, ...,  0.05598235,
        -0.06165052, -0.04068725],
       [ 0.10111272,  0.01626349, -0.05014453, ...,  0.09632763,
         0.04231648, -0.01214947],
       ...,
       [-0.07754684, -0.00684806,  0.14310739, ..., -0.13521372,
        -0.09376252, -0.026759  ],
       [ 0.03269487, -0.026429  , -0.05316353, ...,  0.06118422,
        -0.1858849 , -0.06747157],
       [-0.02768124, -0.03653215, -0.00473706, ..., -0.03963701,
        -0.0155793 , -0.14759911]])

In [42]:
np.allclose(output_slow_1, output_fast_1), \
np.allclose(output_slow_handy_1, output_fast_1), \
np.allclose(output_slow_handy_numba_1, output_fast_1)

(True, True, True)

In [43]:
%%time
K = 5
output_slow_2 = window_correlation(A, K)
print(output_slow_2.shape)
output_slow_2

(1000, 1000)
CPU times: user 25min 2s, sys: 32.7 ms, total: 25min 2s
Wall time: 25min 2s


array([[-0.07439103, -0.01091494,  0.00590264, ...,  0.00420167,
        -0.00604555, -0.06576106],
       [ 0.00127212, -0.03565041, -0.02769068, ..., -0.00401281,
        -0.02822618, -0.04476564],
       [-0.00482727,  0.0002317 ,  0.02088179, ...,  0.01978132,
         0.00488361, -0.01271319],
       ...,
       [ 0.00484049, -0.02124401,  0.04141978, ..., -0.03924662,
        -0.05564318, -0.05646575],
       [ 0.01519657, -0.01927922, -0.00714525, ...,  0.03698401,
        -0.07794346,  0.02738142],
       [-0.03787318, -0.05620067, -0.00999005, ...,  0.01133261,
        -0.03155926, -0.05468683]])

In [44]:
%%time
K = 5
output_slow_handy_2 = window_correlation(A, K)
print(output_slow_handy_2.shape)
output_slow_handy_2

(1000, 1000)
CPU times: user 25min 46s, sys: 38.1 ms, total: 25min 46s
Wall time: 25min 46s


array([[-0.07439103, -0.01091494,  0.00590264, ...,  0.00420167,
        -0.00604555, -0.06576106],
       [ 0.00127212, -0.03565041, -0.02769068, ..., -0.00401281,
        -0.02822618, -0.04476564],
       [-0.00482727,  0.0002317 ,  0.02088179, ...,  0.01978132,
         0.00488361, -0.01271319],
       ...,
       [ 0.00484049, -0.02124401,  0.04141978, ..., -0.03924662,
        -0.05564318, -0.05646575],
       [ 0.01519657, -0.01927922, -0.00714525, ...,  0.03698401,
        -0.07794346,  0.02738142],
       [-0.03787318, -0.05620067, -0.00999005, ...,  0.01133261,
        -0.03155926, -0.05468683]])

In [45]:
%%time
K = 5
output_fast_2 = trace_correlation(A=A, K=K)
print(output_fast_2.shape)
output_fast_2

/tmp/ipykernel_73086/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 8.56 s, sys: 5.31 s, total: 13.9 s
Wall time: 13.9 s


array([[-0.07439103, -0.01091494,  0.00590264, ...,  0.00420167,
        -0.00604555, -0.06576106],
       [ 0.00127212, -0.03565041, -0.02769068, ..., -0.00401281,
        -0.02822618, -0.04476564],
       [-0.00482727,  0.0002317 ,  0.02088179, ...,  0.01978132,
         0.00488361, -0.01271319],
       ...,
       [ 0.00484049, -0.02124401,  0.04141978, ..., -0.03924662,
        -0.05564318, -0.05646575],
       [ 0.01519657, -0.01927922, -0.00714525, ...,  0.03698401,
        -0.07794346,  0.02738142],
       [-0.03787318, -0.05620067, -0.00999005, ...,  0.01133261,
        -0.03155926, -0.05468683]])

In [46]:
np.allclose(output_slow_2, output_slow_handy_2), np.allclose(output_slow_2, output_fast_2)

(True, True)

### Выводы

Расчет корреляции, реализованный руками, работает быстрее np.corrcoef

Numba сильно ускоряет решение через циклы и работает даже быстрее, чем полностью векторизованное решение

### Fast solution on K from 7 to 13

In [47]:
%%time
K = 7
output = trace_correlation(A=A, K=K)
print(output.shape)
output

/tmp/ipykernel_73086/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 18.3 s, sys: 12 s, total: 30.3 s
Wall time: 30.3 s


array([[-0.09516077,  0.01670033, -0.03024807, ...,  0.04411597,
         0.03050622, -0.03028918],
       [ 0.03097562, -0.04038865,  0.01898884, ...,  0.04618197,
        -0.0143901 , -0.04351814],
       [ 0.01087913, -0.024528  ,  0.02240769, ...,  0.00134656,
        -0.00649848,  0.00302833],
       ...,
       [ 0.00622831, -0.00940435,  0.03821958, ..., -0.02575467,
        -0.04810247, -0.03920034],
       [ 0.02772647, -0.00373594, -0.00778935, ...,  0.01049987,
        -0.03917142,  0.00375682],
       [ 0.02513027, -0.06879507, -0.04698267, ...,  0.01042809,
        -0.00530801,  0.00739682]])

In [48]:
%%time
K = 9
output = trace_correlation(A=A, K=K)
print(output.shape)
output

/tmp/ipykernel_73086/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 32.3 s, sys: 23 s, total: 55.3 s
Wall time: 55.3 s


array([[-0.02569709,  0.02094336, -0.02982512, ...,  0.03977397,
         0.02417163,  0.00080713],
       [ 0.02979165, -0.05461423,  0.01231522, ...,  0.04581913,
        -0.0160311 , -0.0469382 ],
       [ 0.01972444, -0.0329929 ,  0.01750263, ..., -0.02468083,
        -0.00299521, -0.00166587],
       ...,
       [ 0.01538205,  0.00277362,  0.03290355, ..., -0.03009591,
        -0.01525593, -0.00668925],
       [ 0.00692997, -0.0123578 , -0.0286321 , ..., -0.00990469,
        -0.00194792, -0.03138683],
       [-0.0002041 , -0.03637102,  0.00392323, ...,  0.00566267,
         0.02238391, -0.00711073]])

In [49]:
%%time
K = 11
output = trace_correlation(A=A, K=K)
print(output.shape)
output

/tmp/ipykernel_73086/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 53.6 s, sys: 50.7 s, total: 1min 44s
Wall time: 1min 44s


array([[ 0.00602239,  0.02733354, -0.04108482, ...,  0.01115629,
         0.04057001, -0.01289275],
       [ 0.01855462, -0.04926594,  0.02294047, ...,  0.03780313,
        -0.04400239, -0.01570932],
       [ 0.0118941 , -0.03573519,  0.01417571, ..., -0.01850147,
         0.02743644, -0.0001143 ],
       ...,
       [ 0.00759125,  0.01226855,  0.02939927, ..., -0.02959592,
         0.0004894 ,  0.00524642],
       [ 0.01066272, -0.0130236 , -0.03940543, ...,  0.01536507,
        -0.00367286, -0.03655842],
       [-0.00796449, -0.01059514,  0.00280168, ...,  0.00870191,
         0.01251542, -0.05754361]])

In [50]:
%%time
K = 13
output = trace_correlation(A=A, K=K)
print(output.shape)
output

/tmp/ipykernel_73086/2853475461.py:31: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=-1, keepdims=True)) \


(1000, 1000)
CPU times: user 3min 38s, sys: 8min 31s, total: 12min 9s
Wall time: 3min 4s


array([[ 0.01027976,  0.02849873, -0.02471407, ..., -0.0222223 ,
         0.0513137 ,  0.00730063],
       [ 0.0096761 , -0.02962917,  0.03296373, ...,  0.01963609,
        -0.02791777, -0.01939336],
       [-0.00939707, -0.0189906 ,  0.00686442, ..., -0.0120754 ,
         0.03200732, -0.00390682],
       ...,
       [-0.00709583,  0.01967984,  0.01672797, ..., -0.03475918,
        -0.01025726, -0.01214221],
       [ 0.00944531, -0.0116479 , -0.0337406 , ...,  0.00497418,
        -0.01546326, -0.03241965],
       [-0.0286403 , -0.00866098,  0.01337691, ...,  0.00255425,
         0.01073406, -0.05612152]])